# CausalMan: Observational Dataset Generation Across Seeds

This notebook generates one observational dataset for every seed in `SEEDS`,
requesting exactly `N_SAMPLES` rows for each dataset. The dataset returned by
`CausalMan.sample()` is saved directly.


## 1. Configuration

Edit only this cell for a standard generation run.

In [ ]:
# ── The only cell you normally need to edit ──────────────────────────────────

CHOICE = "causalman_micro"       # micro | small | medium | large
SEEDS = [4, 6, 42, 66, 90]      # one dataset is generated for each seed
N_SAMPLES = 10_000               # rows returned in each observational dataset

OUTPUT_ROOT = "output/causalman_observational"
PARALLELIZE = True
MAX_WORKERS = 5
DEBUG_MODE = False

# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports and output directory

In [ ]:
import os
from datetime import datetime
from pathlib import Path
import sys

# Put the repository root before this notebook directory so causalman.py
# cannot shadow the causalman package when the notebook runs in-place.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "causalman" / "__init__.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is not None:
    project_root = str(PROJECT_ROOT)
    if project_root in sys.path:
        sys.path.remove(project_root)
    sys.path.insert(0, project_root)

from causalman import CausalMan

timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
run_dir = os.path.join(OUTPUT_ROOT, f"{CHOICE}_{timestamp}")
os.makedirs(run_dir, exist_ok=True)

print(f"Results will be saved to: {os.path.abspath(run_dir)}")


## 3. Generate one observational dataset per seed

For every seed, the notebook asks CausalMan for exactly `N_SAMPLES` rows and
saves the returned observational dataset and aligned interventional table.

In [ ]:
generation_summary = []

for seed in SEEDS:
    print(f"\n── {CHOICE} | seed={seed} ──")

    seed_dir = os.path.join(run_dir, f"seed_{seed:03d}")
    simulator_dir = os.path.join(seed_dir, "simulator_output")
    os.makedirs(simulator_dir, exist_ok=True)

    simulator = CausalMan(
        name=CHOICE,
        seed=seed,
        parallelize=PARALLELIZE,
        max_workers=MAX_WORKERS,
        debug_mode=DEBUG_MODE,
        save_path=simulator_dir,
    )

    obs_dataset, _, _, _, _, interventional_table = simulator.sample(
        n_samples=N_SAMPLES
    )

    dataset_path = os.path.join(
        seed_dir,
        f"{CHOICE}_observational_n{N_SAMPLES}_RS{seed}.csv",
    )
    obs_dataset.to_csv(dataset_path, index=False)

    interventional_table_path = os.path.join(
        seed_dir, f"{CHOICE}_interventional_table.csv"
    )
    interventional_table.to_csv(interventional_table_path, index=False)

    generation_summary.append(
        {
            "seed": seed,
            "rows": len(obs_dataset),
            "variables": obs_dataset.shape[1],
            "dataset_path": dataset_path,
        }
    )

    print(f"Rows:       {len(obs_dataset):,}")
    print(f"Variables:  {obs_dataset.shape[1]}")
    print(f"Dataset:    {dataset_path}")

## 4. Generation summary

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(generation_summary)
display(summary_df)

summary_path = os.path.join(run_dir, "generation_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary to: {summary_path}")

## Output structure

```text
OUTPUT_ROOT/
└── causalman_<scale>_<timestamp>/
    ├── generation_summary.csv
    ├── seed_004/
    │   ├── causalman_<scale>_observational_n<N>_RS4.csv
    │   ├── causalman_<scale>_interventional_table.csv
    │   └── simulator_output/
    └── ...
```
